# AlphaFold3 GPU Benchmark (T4, free Colab)

Runs AlphaFold3 on the **same 118-residue toy sequence, empty MSA, seed=1**
as `af3_cpu_colab.ipynb` and the existing CPU run in `results/sweep/`, so
the CPU/GPU/TPU comparison in `results/sweep/af3_comparison.md` is
apples-to-apples on input, only the backend changes.

**Before running:** go to `Runtime > Change runtime type > T4 GPU`, then
run all cells in order.


## 1. Confirm the runtime matches this notebook

In [1]:
!nvidia-smi

Wed Aug 12 05:53:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. System dependencies (Colab has root, unlike the Stanford login node -- AF3 needs a real C++ toolchain to build `libcifpp`/pybind11 at install time)

In [2]:
!apt-get -qq update && apt-get -qq install -y --no-install-recommends \
    build-essential cmake ninja-build zlib1g-dev libeigen3-dev \
    libpcre2-dev libboost-all-dev libbz2-dev zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libboost1.74-tools-dev.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../000-libboost1.74-tools-dev_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost1.74-tools-dev (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-tools-dev.
Preparing to unpack .../001-libboost-tools-dev_1.74.0.3ubuntu7_amd64.deb ...
Unpacking libboost-tools-dev (1.74.0.3ubuntu7) ...
Selecting previously unselected package libboost-atomic1.74.0:amd64.
Preparing to unpack .../002-libboost-atomic1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-atomic1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libboost-atomic1.74-dev:amd64.
Preparing to unpack .../0

## 3. Install `uv` and clone AlphaFold3

In [3]:
!curl -fsSL https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

!git clone --depth 1 https://github.com/google-deepmind/alphafold3.git /content/alphafold3

downloading uv 0.12.3 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into '/content/alphafold3'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (193/193), done.
remote: Total 235 (delta 43), reused 145 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (235/235), 5.68 MiB | 20.18 MiB/s, done.
Resolving deltas: 100% (43/43), done.


## 4. Build AlphaFold3 (`uv sync` compiles `libcifpp` -- this is the step that needs the toolchain from step 2 -- then `uv run build_data` processes the Chemical Component Dictionary into the pickle `run_alphafold.py` expects at startup. Both steps are in DeepMind's own docker/Dockerfile; skipping the second one produces a FileNotFoundError for chemical_component_sets.pickle a few seconds into Step 7, which is exactly what happened on this project's first attempt.)

In [4]:
%cd /content/alphafold3
!uv sync
!uv run build_data

/content/alphafold3
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 106 packages in 2ms
Prepared 70 packages in 10m 15s
Installed 70 packages in 370ms
 + absl-py==2.3.1
 + aiofiles==25.1.0
 + alphafold3==3.0.3.dev1+g29596b970 (from file:///content/alphafold3)
 + annotated-types==0.7.0
 + chex==0.1.91
 + dm-haiku==0.0.16
 + einshape==1.0
 + etils==1.13.0
 + flax==0.12.2
 + fsspec==2026.6.0
 + humanize==4.15.0
 + immutabledict==4.2.2
 + importlib-resources==6.5.2
 + iniconfig==2.3.0
 + jax==0.10.2
 + jax-cuda12-pjrt==0.10.2
 + jax-cuda12-plugin==0.10.2
 + jaxlib==0.10.2
 + jaxtyping==0.3.5
 + jmp==0.0.4
 + markdown-it-py==4.0.0
 + mdurl==0.1.2
 + ml-dtypes==0.5.4
 + msgpack==1.1.2
 + nest-asyncio==1.6.0
 + numpy==2.4.1
 + nvidia-cublas-cu12==12.9.1.4
 + nvidia-cuda-cccl-cu12==12.9.27
 + nvidia-cuda-cupti-cu12==12.9.79
 + nvidia-cuda-nvcc-cu12==12.9.86
 + nvidia-cuda-nvrtc-cu12==12.9.86
 + nvidia-cuda-runtime-cu12==12.9.79
 + nvidia-c

## 5. Download + decompress AF3 weights (~1GB compressed / ~1.15GB decompressed; public direct download, no approval form, subject to DeepMind's Weights Terms of Use)

In [5]:
!mkdir -p /content/af3_weights
!curl -fsSL -o /content/af3_weights/af3.bin.zst \
    https://storage.googleapis.com/alphafold3/af3.bin.zst
!zstd -d /content/af3_weights/af3.bin.zst -o /content/af3_weights/af3.bin

/content/af3_weights/af3.bin.zst: 1146811260 bytes 


## 6. Build the input JSON -- same 118-residue toy sequence used throughout this project, empty MSA, seed=1 (identical to `src/make_af3_input.py`, so this run is directly comparable to the CPU/TPU runs)

In [6]:
import json

TOY_SEQUENCE_118 = (
    "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKV"
    "KALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWD"
)

payload = {
    "name": "af3_toy_test",
    "modelSeeds": [1],
    "sequences": [
        {
            "protein": {
                "id": "A",
                "sequence": TOY_SEQUENCE_118,
                "unpairedMsa": "",
                "pairedMsa": "",
                "templates": [],
            }
        }
    ],
    "dialect": "alphafold3",
    "version": 1,
}

with open("/content/alphafold3/af3_toy_input.json", "w") as f:
    json.dump(payload, f, indent=2)

print(json.dumps(payload, indent=2))

{
  "name": "af3_toy_test",
  "modelSeeds": [
    1
  ],
  "sequences": [
    {
      "protein": {
        "id": "A",
        "sequence": "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWD",
        "unpairedMsa": "",
        "pairedMsa": "",
        "templates": []
      }
    }
  ],
  "dialect": "alphafold3",
  "version": 1
}


## 7. Run it -- tagged `gpu-t4` so it slots into the same comparison table as the CPU/TPU runs

**Runs via `uv run`, not a bare `python3` call:** `uv sync` in step 4 built AlphaFold3 into an isolated `.venv`, invisible to the system Python. Calling `python3 run_alphafold.py` directly fails with `ModuleNotFoundError: No module named 'alphafold3'` in well under a second -- that's not a real (fast!) run, it's an immediate crash. `uv run` executes inside the venv where the package actually lives.

**This cell now fails loudly (an `AssertionError`, cell stops with a red error) if the run didn't actually work -- either a non-zero exit code, or a suspiciously fast wall-clock (<30s). Do not proceed past a red error here.**

In [9]:
import json, subprocess, time, os

output_dir = "/content/af3_output"
cmd = [
    "uv", "run", "python3", "run_alphafold.py",
    "--json_path=/content/alphafold3/af3_toy_input.json",
    "--model_dir=/content/af3_weights",
    f"--output_dir={output_dir}",
    "--norun_data_pipeline",
    "--flash_attention_implementation=xla",
    "--jax_backend=gpu",
]

os.environ["XLA_FLAGS"] = "--xla_disable_hlo_passes=custom-kernel-fusion-rewriter"

t0 = time.time()
proc = subprocess.run(cmd, cwd="/content/alphafold3", capture_output=True, text=True)
elapsed = time.time() - t0
print(proc.stdout[-3000:])
print(proc.stderr[-3000:])
print(f"Total wall-clock: {elapsed:.2f}s")

# Fail LOUDLY instead of silently -- subprocess.run() does not raise on a
# non-zero exit code, so without this check a crashed run_alphafold.py
# (e.g. ModuleNotFoundError if run_alphafold.py is ever called without
# `uv run`) would print a traceback buried in the output above and this
# notebook would carry on to Step 8 as if nothing happened, producing a
# result_af3_*.json full of nulls with no visible error. This exact
# failure mode happened during this project's first attempt at this
# notebook -- do not remove this check.
assert proc.returncode == 0, (
    f"\n\n{'='*70}\nrun_alphafold.py FAILED (exit code {proc.returncode}) "
    f"after {elapsed:.2f}s.\nThis is NOT a successful run -- do not "
    f"proceed to Step 8. See the stderr printed above for the real error."
    f"\n{'='*70}"
)
assert elapsed > 30, (
    f"\n\n{'='*70}\nrun_alphafold.py returned exit code 0 but took only "
    f"{elapsed:.2f}s -- too fast to be a real 5-sample run (the existing "
    f"CPU result in this project took ~392s). Treat this as a failure and "
    f"inspect the output above before trusting it.\n{'='*70}"
)
print("\nLooks like a real run -- proceed to Step 8.")


Running AlphaFold 3. Please note that standard AlphaFold 3 model parameters are
only available under terms of use provided at
https://github.com/google-deepmind/alphafold3/blob/main/WEIGHTS_TERMS_OF_USE.md.
If you do not agree to these terms and are using AlphaFold 3 derived model
parameters, cancel execution of AlphaFold 3 inference with CTRL-C, and do not
use the model parameters.

Found local GPU devices: [CudaDevice(id=0)], using device 0: cuda:0
Building model from scratch...
Checking that model parameters can be loaded...

Running fold job af3_toy_test...
Output will be written in /content/af3_output/af3_toy_test
Skipping data pipeline...
Writing model input JSON to /content/af3_output/af3_toy_test/af3_toy_test_data.json
Predicting 3D structure for af3_toy_test with 1 seed(s)...
Featurising data with 1 seed(s)...
Featurising data with seed 1.
Featurising data with seed 1 took 3.21 seconds.
Featurising data with 1 seed(s) took 11.09 seconds.
Running model inference and extracting

## 8. Write the result JSON (same schema as `results/result_cpu-colab.json` / `results/result_gpu-t4.json`, copy this into `results/` in the repo)

In [10]:
import json, glob

summary_path = glob.glob(f"{output_dir}/**/summary_confidences.json", recursive=True)
summary = json.load(open(summary_path[0])) if summary_path else {}

result = {
    "run_tag": "gpu-t4",
    "backend": "cuda",
    "model": "alphafold3",
    "num_residues": 118,
    "num_samples": 5,
    "total_inference_seconds": elapsed,
    "seconds_per_sample": elapsed / 5,
    "best_ranking_score": summary.get("ranking_score"),
    "ptm": summary.get("ptm"),
    "fraction_disordered": summary.get("fraction_disordered"),
    "has_clash": summary.get("has_clash"),
}

result_path = "/content/result_af3_gpu-t4.json"
json.dump(result, open(result_path, "w"), indent=2)
print(json.dumps(result, indent=2))

{
  "run_tag": "gpu-t4",
  "backend": "cuda",
  "model": "alphafold3",
  "num_residues": 118,
  "num_samples": 5,
  "total_inference_seconds": 114.11956882476807,
  "seconds_per_sample": 22.823913764953613,
  "best_ranking_score": null,
  "ptm": null,
  "fraction_disordered": null,
  "has_clash": null
}


## 9. Print it (copy into `results/` in the repo) and download the result + the raw AF3 output files

In [12]:
!cat /content/result_af3_gpu-t4.json

{
  "run_tag": "gpu-t4",
  "backend": "cuda",
  "model": "alphafold3",
  "num_residues": 118,
  "num_samples": 5,
  "total_inference_seconds": 114.11956882476807,
  "seconds_per_sample": 22.823913764953613,
  "best_ranking_score": null,
  "ptm": null,
  "fraction_disordered": null,
  "has_clash": null
}

In [14]:
from google.colab import files
files.download('/content/result_af3_gpu-t4.json')

# Raw AF3 output -- copy these into structure/ and results/sweep/ the same
# way the existing af3_toy_test_* files were added, then update
# results/sweep/af3_comparison.md Section 3 with the real numbers.
import glob
for pattern in ["**/*.cif", "**/summary_confidences.json", "**/ranking_scores.csv"]:
    for path in glob.glob(f"{output_dir}/{pattern}", recursive=True):
        print("Found:", path)
        files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: /content/af3_output/af3_toy_test/af3_toy_test_model.cif


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Found: /content/af3_output/af3_toy_test/seed-1_sample-1/af3_toy_test_seed-1_sample-1_model.cif


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import glob, json

summary_path = glob.glob(f"{output_dir}/**/*summary_confidences.json", recursive=True)
ranking_path = glob.glob(f"{output_dir}/**/*ranking_scores.csv", recursive=True)
print("summary:", summary_path)
print("ranking:", ranking_path)

if summary_path:
    print(open(summary_path[0]).read())
if ranking_path:
    print(open(ranking_path[0]).read())

summary: ['/content/af3_output/af3_toy_test/af3_toy_test_summary_confidences.json', '/content/af3_output/af3_toy_test/seed-1_sample-4/af3_toy_test_seed-1_sample-4_summary_confidences.json', '/content/af3_output/af3_toy_test/seed-1_sample-0/af3_toy_test_seed-1_sample-0_summary_confidences.json', '/content/af3_output/af3_toy_test/seed-1_sample-2/af3_toy_test_seed-1_sample-2_summary_confidences.json', '/content/af3_output/af3_toy_test/seed-1_sample-3/af3_toy_test_seed-1_sample-3_summary_confidences.json', '/content/af3_output/af3_toy_test/seed-1_sample-1/af3_toy_test_seed-1_sample-1_summary_confidences.json']
ranking: ['/content/af3_output/af3_toy_test/af3_toy_test_ranking_scores.csv']
{
 "chain_ids": [
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "A",
  "